# 01 - Build the hybrid dataset (E01-09, #27)
Provenance, license-separated namespaces, deterministic dedup + grouped splits, label-leak audit, contract-pinned features.
**Parameters** (env): `SQLTOVIZ_MODE` smoke|full, `SQLTOVIZ_COMMIT`, `SQLTOVIZ_DATA`, `SQLTOVIZ_OUT`. Code is Apache-2.0; Spider/BIRD material stays in `cc-by-sa-4.0/` (CC BY-SA 4.0); synthetic data in `apache-2.0/`.
Labels: synthetic (by construction) and LLM pilot labels are kept separate; there are **no human labels yet** (100 ids reserved).

In [ ]:
import os, sys, json, subprocess, pathlib, time
MODE = os.environ.get("SQLTOVIZ_MODE", "smoke")            # "smoke" (minutes, subset) | "full"
SEED = 20261127
COMMIT = os.environ.get("SQLTOVIZ_COMMIT", "development")   # pin a commit SHA for publishable runs
REPO = os.environ.get("SQLTOVIZ_REPO")
if not REPO:                                                # Kaggle: clone the repository at the pinned commit
    REPO = "/kaggle/working/sqltoviz"
    if not os.path.exists(REPO):
        subprocess.run(["git", "clone", "https://github.com/jmmana/sqltoviz", REPO], check=True)
        subprocess.run(["git", "-C", REPO, "checkout", COMMIT], check=True)
sys.path.insert(0, f"{REPO}/research/kaggle")
OUT = os.environ.get("SQLTOVIZ_OUT", "/kaggle/working/out"); os.makedirs(OUT, exist_ok=True)
DATA = os.environ.get("SQLTOVIZ_DATA", "/kaggle/working/data")
GIT = subprocess.run(["git", "-C", REPO, "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("mode", MODE, "| repo commit", GIT, "| out", OUT)
from sqltoviz_data import features as F
F.load_contract()                                           # raises if the contract bytes differ from the pinned SHA256
print(F.CONTRACT_VERSION, F.CONTRACT_SHA256)

## 1. Acquire Spider / BIRD (checksum-verified; skipped if already present)

In [ ]:
import pathlib
from sqltoviz_data import acquire, pipeline as P, splits as S, synthgen as G, leakage as L, experiment as E
NS = f"{DATA}/cc-by-sa-4.0"
acq = []
if not (os.path.exists(f"{NS}/spider/spider_data/dev.json") and os.path.exists(f"{NS}/bird_dev")):
    man = acquire.load_manifest(pathlib.Path(REPO) / "research/kaggle/manifests/sources.json")
    acq = [acquire.acquire(n, man[n], pathlib.Path(DATA)) for n in ("spider", "bird_dev")]
print(json.dumps(acq, indent=1) if acq else "sources already present at " + NS)

## 2. Real candidates: execute gold SQL in the read-only sandbox, keep within-contract results

In [ ]:
MAXS = 700 if MODE == "smoke" else None
real, real_stats = P.real_candidates(NS, max_per_source=MAXS)
print(json.dumps(real_stats, indent=1)); print(len(real), "executable items")

## 3. Deduplicate (within database) and split by database (no schema in two splits)

In [ ]:
kept, dropped, dstats = S.dedup(real, group=lambda x: x["db_id"])
for it in kept: it["label"] = "unlabelled"             # real items have no chart label here; labels come from annotation
assign = S.grouped_stratified_split(kept, label=lambda x: x["label"], group=lambda x: x["db_id"], seed=SEED)
audit = S.audit_splits(kept, assign, label=lambda x: x["label"], group=lambda x: x["db_id"])   # raises on leakage
print(dstats); print({k: (v["n"], v["groups"]) for k, v in audit["splits"].items()})

## 4. Synthetic data (class-agnostic schemas/templates) + measured label-leakage audit

In [ ]:
NPC = 150 if MODE == "smoke" else 2000
syn = G.generate(NPC, SEED)
G.assert_no_chart_vocabulary(syn)
leak = L.report(syn, seed=0)
print("structural ceiling", leak["structural_ceiling_accuracy"])
for k, v in leak["views"].items(): print(f"{k:26s} acc={v['accuracy']:.3f} macroF1={v['macro_f1']:.3f}")
assert leak["views"]["sql_features16"]["accuracy"] <= leak["structural_ceiling_accuracy"] + 0.05, "SQL-only leakage above structural ceiling"
assert leak["views"]["names_only"]["accuracy"] <= leak["structural_ceiling_accuracy"] + 0.05

## 5. Contract-pinned features and artifacts (license namespaces kept apart)

In [ ]:
for d in ("apache-2.0", "cc-by-sa-4.0", "llm-labels", "reports"): os.makedirs(f"{OUT}/{d}", exist_ok=True)
with open(f"{OUT}/apache-2.0/synthetic.jsonl", "w") as f:
    for it in syn: f.write(json.dumps(it, sort_keys=True) + "\n")
fs = P.save_features(f"{OUT}/apache-2.0/features_synthetic.npz", syn)
with open(f"{OUT}/cc-by-sa-4.0/real_items.jsonl", "w") as f:
    for it in kept: f.write(json.dumps(it, default=str) + "\n")
fr = P.save_features(f"{OUT}/cc-by-sa-4.0/features_real.npz", kept)
json.dump(assign, open(f"{OUT}/cc-by-sa-4.0/real_splits.json", "w"))
import shutil
lab = pathlib.Path(REPO) / "research/kaggle/annotations"
for n in ("pilot_llm_labels.jsonl", "human_validation_reserve_ids.json"):
    if (lab / n).exists(): shutil.copy(lab / n, f"{OUT}/llm-labels/{n}")
print(fs, fr)

In [ ]:
report = {"mode": MODE, "seed": SEED, "repo_commit": GIT, "contract_version": F.CONTRACT_VERSION, "contract_sha256": F.CONTRACT_SHA256, "interpretations": F.interpretations_pin(REPO),
          "acquisition": acq, "real_candidates": real_stats, "dedup": dstats, "real_split_audit": audit, "synthetic_leakage_audit": leak,
          "synthetic_sha256": E.sha256_items(syn), "feature_files": {"synthetic": fs, "real": fr},
          "label_provenance": {"synthetic": "by construction", "llm": "pilot only (llm-labels/)", "human": "none yet"}}
json.dump(report, open(f"{OUT}/reports/dataset_audit.json", "w"), indent=1, default=str)
files = {"apache-2.0/synthetic.jsonl": "apache-2.0", "apache-2.0/features_synthetic.npz": "apache-2.0",
         "cc-by-sa-4.0/real_items.jsonl": "cc-by-sa-4.0", "cc-by-sa-4.0/features_real.npz": "cc-by-sa-4.0", "cc-by-sa-4.0/real_splits.json": "cc-by-sa-4.0",
         "reports/dataset_audit.json": "apache-2.0"}
for n in ("pilot_llm_labels.jsonl", "human_validation_reserve_ids.json"):
    if os.path.exists(f"{OUT}/llm-labels/{n}"): files[f"llm-labels/{n}"] = "cc-by-sa-4.0"
man = P.write_manifest(OUT, files, {"mode": MODE, "seed": SEED, "repo_commit": GIT})
print(json.dumps({k: v["sha256"][:12] for k, v in man["files"].items()}, indent=1))